[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-03-advanced-sql.ipynb#scrollTo=a1c2e3f4)

---
# Day 3 · Advanced SQL in DuckDB — Window Functions, CTEs, and Macros
**certified-journeys / duckdb-certified** · Day 3 · Advanced SQL

> **Goal for today:** Master DuckDB's advanced SQL capabilities — window functions with `QUALIFY`, readable multi-level CTEs, DuckDB-specific extensions (`PIVOT`, `UNPIVOT`, list comprehensions), and reusable SQL macros.


In [ ]:
%pip install -q duckdb


## Setup · Create the Sales Dataset

All examples in this notebook use a synthetic `sales` table with 100 000 rows spanning multiple regions, products, and time periods.


In [ ]:
import duckdb

con = duckdb.connect()

# Build a realistic sales table with regions, products, reps, and monthly variation
con.execute("""
CREATE TABLE sales AS
SELECT
    i                                                     AS sale_id,
    date '2023-01-01' + INTERVAL (i % 365) DAY            AS sale_date,
    ['North','South','East','West'][1 + (i % 4)]          AS region,
    ['Widgets','Gadgets','Gizmos','Doohickeys'][1 + (i % 4)] AS product,
    'Rep_' || lpad(CAST(1 + (i % 10) AS VARCHAR), 2, '0') AS sales_rep,
    round(50 + random() * 950, 2)                         AS amount,
    round(20 + random() * 40, 2)                          AS cost
FROM range(1, 100_001) t(i)
""")

n = con.execute("SELECT count(*) FROM sales").fetchone()[0]
print(f"Table ready: {n:,} rows")
print(con.execute("SELECT * FROM sales LIMIT 4").df().to_string(index=False))


## Step 1 · Window Functions — `ROW_NUMBER`, `RANK`, and `LAG`

Window functions compute a value for each row **relative to a set of related rows** (the "window") without collapsing them into groups.

**Anatomy of a window function:**
```sql
FUNCTION() OVER (
    PARTITION BY col1           -- defines the window group
    ORDER BY col2 DESC          -- defines row ordering within the window
    ROWS BETWEEN ... AND ...    -- optional frame clause
)
```

| Function | Returns | Use case |
|---|---|---|
| `ROW_NUMBER()` | Unique 1-N within partition | De-duplication, pagination |
| `RANK()` | 1-N with gaps on ties | Leaderboards |
| `DENSE_RANK()` | 1-N without gaps | Rankings without holes |
| `LAG(col, n)` | Value from N rows before | Period-over-period diff |
| `LEAD(col, n)` | Value from N rows after | Forecasting look-ahead |
| `SUM() OVER (ORDER BY ...)` | Cumulative sum | Running totals |
| `FIRST_VALUE()` | First value in window | Baseline comparisons |

> **Reading:** [DuckDB Window Functions](https://duckdb.org/docs/sql/window_functions)


In [ ]:
# ROW_NUMBER, RANK, and LAG over a monthly sales aggregation
# First: build a monthly summary CTE, then apply window functions on top
df_window = con.execute("""
WITH monthly AS (
    -- Aggregate to monthly totals per region
    SELECT
        region,
        strftime(sale_date, '%Y-%m')      AS month,
        round(SUM(amount), 2)             AS revenue
    FROM sales
    GROUP BY region, month
)
SELECT
    region,
    month,
    revenue,
    -- ROW_NUMBER: unique sequential rank per region (no ties)
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY revenue DESC) AS row_num,
    -- RANK: rank with gaps on ties
    RANK()       OVER (PARTITION BY region ORDER BY revenue DESC) AS rnk,
    -- LAG: previous month's revenue for the same region
    LAG(revenue, 1) OVER (PARTITION BY region ORDER BY month)    AS prev_month_revenue,
    -- Derived: month-over-month change
    round(
        revenue - LAG(revenue, 1) OVER (PARTITION BY region ORDER BY month),
    2) AS mom_delta
FROM monthly
ORDER BY region, month
LIMIT 12
""").df()

print(df_window.to_string(index=False))


### What just happened?
- **`PARTITION BY region`** creates a separate window per region — `ROW_NUMBER` resets to 1 for each region.
- **`ROW_NUMBER`** guarantees uniqueness even on ties; **`RANK`** assigns the same value to ties but skips subsequent ranks.
- **`LAG(revenue, 1)`** looks back exactly 1 row within the `ORDER BY month` ordering — the first row per partition gets `NULL` (no prior month).
- Computing `mom_delta` inline avoids a self-join — this is the primary ergonomic win of window functions.


## Step 2 · `QUALIFY` — Filtering on Window Results Without a Subquery

`QUALIFY` is a DuckDB (and Snowflake) extension that filters rows based on window function results — analogous to `HAVING` for aggregations, but for window functions.

**Without `QUALIFY`** (standard SQL):
```sql
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY region ORDER BY revenue DESC) AS rn
    FROM monthly
) WHERE rn = 1
```

**With `QUALIFY`** (DuckDB):
```sql
SELECT *
FROM monthly
QUALIFY ROW_NUMBER() OVER (PARTITION BY region ORDER BY revenue DESC) = 1
```

The `QUALIFY` clause runs **after** window functions and **after** `WHERE`/`GROUP BY`/`HAVING`, so it can reference any window function in the `SELECT` list or define a new one inline.


In [ ]:
# QUALIFY: keep only the top-2 sales reps per region by total revenue
# Without QUALIFY, this would require a subquery or CTE + filter
top_reps = con.execute("""
SELECT
    region,
    sales_rep,
    round(SUM(amount), 2)   AS total_revenue,
    COUNT(*)                AS deals,
    DENSE_RANK() OVER (PARTITION BY region ORDER BY SUM(amount) DESC) AS dr
FROM sales
GROUP BY region, sales_rep
QUALIFY DENSE_RANK() OVER (PARTITION BY region ORDER BY SUM(amount) DESC) <= 2
ORDER BY region, dr
""").df()

print("Top 2 reps per region (via QUALIFY):")
print(top_reps.to_string(index=False))

# QUALIFY to de-duplicate: keep only the most recent record per sale_id
# (simulating a late-arriving corrections scenario)
deduplicated = con.execute("""
SELECT sale_id, sale_date, region, amount
FROM sales
QUALIFY ROW_NUMBER() OVER (PARTITION BY sale_id ORDER BY sale_date DESC) = 1
LIMIT 5
""").df()
print("\nDe-duplicated rows (most recent per sale_id):")
print(deduplicated.to_string(index=False))


### What just happened?
- **`QUALIFY ... <= 2`** filtered the grouped result to only top-2 reps per region — no subquery, no CTE needed.
- The window function in `QUALIFY` can be **different** from those in `SELECT` — you can filter on `ROW_NUMBER()` while selecting `DENSE_RANK()`.
- **De-duplication with `QUALIFY`** is a common production pattern: keep the latest revision of each record without a `NOT IN` subquery.
- `QUALIFY` runs in the logical order: `WHERE` → `GROUP BY` → `HAVING` → window functions → `QUALIFY` → `SELECT` → `ORDER BY` → `LIMIT`.


## Step 3 · CTEs — Readable Multi-Level Query Composition

**Common Table Expressions (CTEs)** are named subqueries defined with `WITH`. DuckDB fully supports:
- **Multiple CTEs** in one `WITH` clause (comma-separated)
- **Recursive CTEs** (`WITH RECURSIVE`) for tree traversal
- **Materialized CTEs** (`WITH ... AS MATERIALIZED`) to force evaluation

**When to use CTEs vs subqueries:**
- CTEs: when the same subquery is referenced multiple times, or when readability matters
- Subqueries: for simple one-off inline filtering
- CTEs are **not always faster** — DuckDB may inline them; use `AS MATERIALIZED` to force caching

> **Reading:** [DuckDB CTEs and Recursive CTEs](https://duckdb.org/docs/sql/query_syntax/with)


In [ ]:
# Multi-level CTE: refactor a complex nested subquery into readable steps
# Goal: find regions where the top sales rep outperforms the regional average by >20%

result = con.execute("""
-- Level 1: aggregate raw sales to rep-level totals
WITH rep_totals AS (
    SELECT
        region,
        sales_rep,
        round(SUM(amount), 2)   AS rep_revenue,
        COUNT(*)                AS deal_count
    FROM sales
    GROUP BY region, sales_rep
),

-- Level 2: compute regional averages from the rep totals
regional_stats AS (
    SELECT
        region,
        round(AVG(rep_revenue), 2)  AS avg_rep_revenue,
        round(MAX(rep_revenue), 2)  AS max_rep_revenue,
        COUNT(*)                    AS rep_count
    FROM rep_totals
    GROUP BY region
),

-- Level 3: join back to find top reps, compute outperformance ratio
top_reps AS (
    SELECT
        r.region,
        r.sales_rep,
        r.rep_revenue,
        s.avg_rep_revenue,
        round(r.rep_revenue / s.avg_rep_revenue, 3)  AS outperformance_ratio
    FROM rep_totals  r
    JOIN regional_stats s ON r.region = s.region
    WHERE r.rep_revenue = s.max_rep_revenue  -- pick the top rep in each region
)

-- Final: filter to regions where top rep beats average by >20%
SELECT *
FROM top_reps
WHERE outperformance_ratio > 1.20
ORDER BY outperformance_ratio DESC
""").df()

print("Regions with a standout top rep (>20% above regional average):")
print(result.to_string(index=False))


### What just happened?
- Three CTEs built a **query pipeline**: raw aggregation → regional stats → per-rep enrichment.
- Each CTE reads the result of the previous one — data flows top-to-bottom, matching how analysts think.
- DuckDB **inlines** non-materialized CTEs by default (like views), so there's no intermediate storage overhead unless you add `AS MATERIALIZED`.
- The final `WHERE outperformance_ratio > 1.20` is evaluated in a single pass over the `top_reps` CTE result — no re-scanning the original table.


## Step 4 · Recursive CTEs — Hierarchy Traversal

`WITH RECURSIVE` lets a CTE reference itself, enabling traversal of hierarchical data (org charts, bill-of-materials, file trees).

**Structure:**
```sql
WITH RECURSIVE cte(cols) AS (
    -- Anchor: starting point (non-recursive)
    SELECT ...  
    UNION ALL
    -- Recursive: join cte against the base table
    SELECT ... FROM base_table JOIN cte ON ...
)
SELECT * FROM cte
```

> **DuckDB tip:** Add `WHERE depth < N` in the recursive branch to guard against infinite loops in cyclic graphs.


In [ ]:
# Build a small org chart and traverse it recursively
con.execute("""
CREATE TABLE IF NOT EXISTS org (
    emp_id   INT,
    emp_name VARCHAR,
    manager_id INT
)
""")
con.execute("DELETE FROM org")  # reset for idempotency
con.executemany("INSERT INTO org VALUES (?, ?, ?)", [
    (1, 'CEO',      None),
    (2, 'VP Sales', 1),
    (3, 'VP Eng',   1),
    (4, 'AE East',  2),
    (5, 'AE West',  2),
    (6, 'SWE 1',    3),
    (7, 'SWE 2',    3),
    (8, 'SWE 3',    3),
])

# Recursive CTE: walk from CEO down to all reports
hierarchy = con.execute("""
WITH RECURSIVE reporting_chain(emp_id, emp_name, manager_id, depth, path) AS (
    -- Anchor: start at the CEO (no manager)
    SELECT emp_id, emp_name, manager_id, 0, CAST(emp_name AS VARCHAR)
    FROM org
    WHERE manager_id IS NULL

    UNION ALL

    -- Recursive: find all direct reports of the current node
    SELECT
        o.emp_id,
        o.emp_name,
        o.manager_id,
        rc.depth + 1,
        rc.path || ' → ' || o.emp_name   -- build path string
    FROM org o
    JOIN reporting_chain rc ON o.manager_id = rc.emp_id
    WHERE rc.depth < 10   -- safety: prevent infinite loop on cyclic data
)
SELECT depth, emp_id, emp_name, path
FROM reporting_chain
ORDER BY path
""").df()

print(hierarchy.to_string(index=False))


### What just happened?
- The **anchor** selected the CEO (depth 0); each **recursive step** found direct reports and incremented depth.
- **`path`** was built by concatenating names with ` → ` — a lightweight alternative to storing full ancestry arrays.
- **`WHERE rc.depth < 10`** prevents infinite recursion if the data has cycles (always include this guard).
- `UNION ALL` is required (not `UNION`) — deduplication in recursive CTEs is expensive and usually unnecessary.


## Step 5 · DuckDB Extensions — `PIVOT`, `UNPIVOT`, and List Comprehensions

DuckDB adds several SQL extensions that eliminate common boilerplate:

**`PIVOT`** — rotate row values into columns (replaces `CASE WHEN` gymnastics):
```sql
PIVOT table ON pivot_col USING AGG(value_col)
```

**`UNPIVOT`** — melt wide columns into rows (the inverse):
```sql
UNPIVOT table ON (col1, col2) INTO NAME key VALUE val
```

**List comprehensions** — transform arrays inline:
```sql
[x * 2 FOR x IN my_list IF x > 0]
```

> **Reading:** [DuckDB PIVOT Statement](https://duckdb.org/docs/sql/statements/pivot)


In [ ]:
# PIVOT: rotate region rows into columns
# Without PIVOT: would need CASE WHEN for each region
print("=== PIVOT — monthly revenue by region as columns ===")
pivoted = con.execute("""
PIVOT (
    SELECT
        strftime(sale_date, '%Y-%m') AS month,
        region,
        round(SUM(amount), 2)        AS revenue
    FROM sales
    WHERE sale_date BETWEEN '2023-01-01' AND '2023-03-31'
    GROUP BY month, region
)
ON region
USING SUM(revenue)
ORDER BY month
""").df()
print(pivoted.to_string(index=False))

# UNPIVOT: melt the wide pivot result back into rows
print("\n=== UNPIVOT — restore long format ===")
# Write pivoted to a temp table first, then UNPIVOT it
con.execute("CREATE OR REPLACE TABLE pivoted_sales AS " + """
PIVOT (
    SELECT strftime(sale_date, '%Y-%m') AS month, region, round(SUM(amount), 2) AS revenue
    FROM sales WHERE sale_date BETWEEN '2023-01-01' AND '2023-03-31'
    GROUP BY month, region
) ON region USING SUM(revenue) ORDER BY month
""")

unpivoted = con.execute("""
UNPIVOT pivoted_sales
ON (North, South, East, West)
INTO NAME region VALUE revenue
ORDER BY month, region
LIMIT 8
""").df()
print(unpivoted.to_string(index=False))


### What just happened?
- **`PIVOT ... ON region USING SUM(revenue)`** created one column per unique region value — no `CASE WHEN` needed.
- **`UNPIVOT ... ON (North, South, East, West) INTO NAME region VALUE revenue`** reversed it: four columns became two (key + value).
- Both operations are pure SQL — no Python pandas reshape, no `pd.melt()` or `pd.pivot_table()` needed.
- DuckDB supports **dynamic PIVOT** (no column list needed if the values are known at query time), unlike most other SQL engines.


In [ ]:
# List comprehensions and array functions
print("=== List comprehensions and array operations ===")

# Aggregate sales into arrays, then apply list comprehension to transform
arr_demo = con.execute("""
WITH daily AS (
    SELECT
        region,
        strftime(sale_date, '%Y-%m') AS month,
        round(SUM(amount), 2)        AS revenue
    FROM sales
    GROUP BY region, month
),
region_series AS (
    -- Collect monthly revenues into a list per region
    SELECT
        region,
        list(revenue ORDER BY month)              AS monthly_revenues,
        max(revenue)                              AS best_month_revenue
    FROM daily
    GROUP BY region
)
SELECT
    region,
    monthly_revenues,
    -- List comprehension: compute % of best month for each month
    [round(r / best_month_revenue * 100, 1) FOR r IN monthly_revenues]  AS pct_of_best
FROM region_series
ORDER BY region
""").df()

for _, row in arr_demo.iterrows():
    print(f"{row['region']:6s}  pct_of_best: {row['pct_of_best']}")


### What just happened?
- **`list(revenue ORDER BY month)`** is DuckDB's ordered list aggregation — it creates a `LIST` (array) type with elements in month order.
- **`[expr FOR x IN list IF condition]`** is a SQL list comprehension — Python-style syntax inside SQL.
- The comprehension `[round(r / best_month_revenue * 100, 1) FOR r IN monthly_revenues]` mapped each element to a percentage without `UNNEST` + re-aggregation.
- `LIST` columns are first-class in DuckDB — you can pass them to `array_length()`, `list_sum()`, `list_sort()`, `list_filter()`, and dozens of other built-ins.


## Step 6 · SQL Macros — Reusable Query Logic

DuckDB macros let you encapsulate SQL expressions or full queries as **named, reusable functions**:

| Type | Syntax | Returns |
|---|---|---|
| Scalar macro | `CREATE MACRO f(x) AS (expr)` | Single value per row |
| Table macro | `CREATE MACRO f(x) AS TABLE (SELECT ...)` | Result set |

**Macros vs views:**
- Views are fixed queries; macros accept **parameters** — closer to stored procedures but purely read-side.
- Macros are **inlined** at query time — no function call overhead.
- Table macros can replace boilerplate subqueries that differ only in filter values.

```sql
-- Scalar macro with default parameter
CREATE MACRO pct_change(new_val, old_val) AS
    round((new_val - old_val) / old_val * 100, 2);

-- Use it like a built-in function
SELECT pct_change(120, 100);  -- → 20.0
```


In [ ]:
# ── Scalar macros ────────────────────────────────────────────────────────────

# 1. pct_change: compute percentage change between two values
con.execute("""
CREATE OR REPLACE MACRO pct_change(new_val, old_val) AS
    round((new_val - old_val) / NULLIF(old_val, 0) * 100, 2)
""")

# 2. gross_margin: compute margin percentage from revenue and cost
con.execute("""
CREATE OR REPLACE MACRO gross_margin(revenue, cost) AS
    round((revenue - cost) / NULLIF(revenue, 0) * 100, 2)
""")

# 3. tier_label: categorise an amount into revenue tiers
con.execute("""
CREATE OR REPLACE MACRO tier_label(amt) AS
    CASE
        WHEN amt >= 800 THEN 'Enterprise'
        WHEN amt >= 400 THEN 'Mid-Market'
        ELSE 'SMB'
    END
""")

# Use all three macros in a single query — they behave like built-in functions
print("=== Scalar macros in action ===")
macro_result = con.execute("""
SELECT
    region,
    round(SUM(amount), 2)                      AS total_revenue,
    round(SUM(cost), 2)                        AS total_cost,
    gross_margin(SUM(amount), SUM(cost))        AS margin_pct,
    -- Month-over-month change using LAG + pct_change macro
    pct_change(
        SUM(amount),
        LAG(SUM(amount)) OVER (PARTITION BY region ORDER BY month(sale_date))
    )                                           AS mom_pct_change
FROM sales
GROUP BY region, month(sale_date)
QUALIFY month(sale_date) IN (1, 2, 3)
ORDER BY region, month(sale_date)
""").df()
print(macro_result.to_string(index=False))


### What just happened?
- Three macros replaced inline `CASE WHEN`, `round(...)` formulas and `NULLIF` guards — the SELECT now reads like business logic, not arithmetic.
- **`CREATE OR REPLACE MACRO`** is idempotent — safe to re-run in notebooks.
- **`gross_margin(SUM(amount), SUM(cost))`** shows macros accept aggregate expressions as arguments — they're evaluated after aggregation.
- **`NULLIF(old_val, 0)`** inside the macro prevents division-by-zero — encapsulate defensive logic once, use everywhere.


In [ ]:
# Table macro: parameterised query that returns a result set
con.execute("""
CREATE OR REPLACE MACRO region_summary(target_region) AS TABLE
    SELECT
        strftime(sale_date, '%Y-%m')   AS month,
        COUNT(*)                        AS deals,
        round(SUM(amount), 2)           AS revenue,
        round(AVG(amount), 2)           AS avg_deal,
        gross_margin(SUM(amount), SUM(cost)) AS margin_pct
    FROM sales
    WHERE region = target_region
    GROUP BY month
    ORDER BY month
""")

# Call the table macro — use FROM as with any table expression
print("=== Table macro: region_summary('North') ===")
north = con.execute("SELECT * FROM region_summary('North') LIMIT 6").df()
print(north.to_string(index=False))

print("\n=== Same macro for South ===")
south = con.execute("SELECT * FROM region_summary('South') LIMIT 6").df()
print(south.to_string(index=False))


### What just happened?
- **Table macros** are called with `FROM macro_name(args)` — they slot into any position a table can appear, including `JOIN` targets.
- The macro called `gross_margin()` (a scalar macro) — macros can compose other macros.
- The query plan is **identical** to inlining the WHERE clause — the macro is expanded at parse time, not executed as a separate step.
- **Production use case:** define `region_summary`, `product_summary`, `rep_summary` macros once; call them in dashboards, tests, and ad-hoc queries without copy-pasting SQL.


In [ ]:
# Challenge: Full advanced SQL pipeline
#
# Using the 'sales' table in `con`, complete the following in a single query:
#
# 1. Write a CTE 'monthly_rep' that aggregates sales to (region, sales_rep, month)
#    with total revenue and deal count.
#
# 2. Write a CTE 'ranked_reps' that uses DENSE_RANK() OVER (PARTITION BY region, month
#    ORDER BY revenue DESC) to rank reps within each region-month.
#
# 3. Use QUALIFY to keep only the top-1 rep per region per month.
#
# 4. Use the tier_label() macro to add a revenue_tier column.
#
# 5. PIVOT the result so each region becomes a column, values = total revenue,
#    rows = month. Show Q1 2023 only (months 1-3).
#
# Expected output: month | East | North | South | West

# Your solution here:
# result = con.execute("""
#     WITH monthly_rep AS (
#         ...
#     ),
#     ranked_reps AS (
#         ...
#     )
#     PIVOT (
#         SELECT ...
#         FROM ranked_reps
#         QUALIFY ...
#         WHERE ...
#     ) ON region USING SUM(revenue) ORDER BY month
# """).df()
# print(result)


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| Window functions | `ROW_NUMBER` / `RANK` / `LAG` — compute per-row values relative to a window partition |
| `QUALIFY` | Filter on window results without a subquery — runs after `HAVING`, before `SELECT` |
| Multi-level CTEs | Chain up to N CTEs in one `WITH` block; each reads the output of the previous |
| Recursive CTE | `WITH RECURSIVE` for tree traversal; always add a depth guard |
| `PIVOT` / `UNPIVOT` | Rotate rows↔columns natively in SQL — no `CASE WHEN`, no Python reshape |
| List comprehensions | `[expr FOR x IN list IF cond]` — transform array columns inline |
| Scalar macros | Encapsulate expressions; called like built-in functions; inlined at parse time |
| Table macros | Parameterised queries; called with `FROM macro(args)`; compose other macros |

> **Tip:** DuckDB supports QUALIFY — a window-function analogue of HAVING that lets you filter on window results without a subquery. It's a huge readability win once you know it exists.

---
## What's next
**Day 4** → DuckDB extensions and integrations — `httpfs` for S3/GCS, `spatial` for geospatial queries, and connecting to external databases.

Mark Day 3 complete in your [tracker](../index.html).
